# Workout Model

## Importing Libraries

In [272]:
import pandas as pd
import numpy as np
import pickle
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

## Global Configuration

In [273]:
# Define a specific number (Seed) to ensure the code behaves exactly the same way
# every time we run it. This prevents random changes in model scores.
SEED = 26

# Lock the random number generator of NumPy using our Seed.
np.random.seed(SEED)

# Configure Pandas to display ALL columns when printing a dataframe.
# By default, Pandas hides middle columns with "..." if there are too many.
# We disable this limit so we can inspect all our features during debugging.
pd.set_option('display.max_columns', None)

## Preprocessing Data

### Load Data

In [274]:
# Load the CSV file
# This dataset contains the list of exercises (Pushups, Squats, etc.)
# and their attributes (Difficulty, Muscle Group, Equipment).
df_final = pd.read_csv('../../datas/dataset_final.csv')

# Clean Column Names
# Just like with meals, we strip spaces to avoid "KeyError" later.
df_final.columns = df_final.columns.str.strip()

# Verification
print(f"Data Loaded: {df_final.shape[0]} rows, {df_final.shape[1]} columns.")

Data Loaded: 21853 rows, 61 columns.


In [280]:
# The function tells us:
# 1. How many rows and columns we have.
# 2. The Name and Data Type (Dtype) of each column (int, float, object/text).
# 3. "Non-Null Count": If this number is lower than the total rows, we have missing data!
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21853 entries, 0 to 21852
Data columns (total 61 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   User_ID                     21853 non-null  int64  
 1   Age_x                       21853 non-null  int64  
 2   Gender_x                    21853 non-null  object 
 3   Height_cm_x                 21853 non-null  int64  
 4   Initial_Weight_kg_x         21853 non-null  int64  
 5   Initial_BMI_x               21853 non-null  float64
 6   BMI_Category                21853 non-null  object 
 7   Body_Fat_Category_x         21853 non-null  float64
 8   Body_Fat_Percentage         21853 non-null  float64
 9   Goal_x                      21853 non-null  object 
 10  Workout_Frequency_x         21853 non-null  int64  
 11  Average_Duration_Minutes_x  21853 non-null  int64  
 12  level_x                     21853 non-null  object 
 13  Badminton_x                 218

### Encode Data

In [ ]:
# Isolate Unique Users
# We want just ONE row per user to build their "Profile".
cols_profile = ['User_ID', 'Age_x', 'Gender_x', 'Initial_Weight_kg_x', 'Goal_x', 'Workout_Frequency_x', 'level_x']
df_profiles = df_final[cols_profile].drop_duplicates().reset_index(drop=True)

# Encode Categorical Data (Text -> Numbers)
# We initialize encoders for Gender, Goal, and Level.
le_gender = LabelEncoder()
le_goal = LabelEncoder()
le_level = LabelEncoder()
le_environment = LabelEncoder()

df_profiles['Gender_Encoded'] = le_gender.fit_transform(df_profiles['Gender_x'])
df_profiles['Goal_Encoded'] = le_goal.fit_transform(df_profiles['Goal_x'])
df_profiles['level_Encoded'] = le_level.fit_transform(df_profiles['level_x'])


# Define Features for Similarity Search
features_knn = ['Goal_Encoded', 'Workout_Frequency_x', 'level_Encoded', 'Gender_Encoded', 'Age_x', 'Initial_Weight_kg_x']

### Splitting & Scaling Data

In [276]:
# Split the data
# We split the profiles into a "Database" (Train) and "New Users" (Test)
train_df, test_df = train_test_split(
    df_profiles, test_size=0.3, random_state=SEED
)

# Initialize Scaler & Encode
# We fit the scaler ONLY on the "Database" (Train)
scaler = MinMaxScaler()
X_train = scaler.fit_transform(train_df[features_knn])

# We transform the "New Users" (Test) using the Database's rules
X_test = scaler.transform(test_df[features_knn])

## Modelling

### Model

In [277]:
# Train KNN on the Database
knn = NearestNeighbors(n_neighbors=1, metric='euclidean')
knn.fit(X_train)

,n_neighbors,1
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'euclidean'
,p,2
,metric_params,None
,n_jobs,None


### Evaluation

In [278]:
print("Calculating Metric Evaluation...")
distances, indices = knn.kneighbors(X_test)
eval_data = []

# Evaluation Loop
count = len(test_df)

for i in range(count):
    # The "Target" (New User from Test Data)
    user = test_df.iloc[i]
    
    # The "Recommendation" (Existing User from Train Data)
    # indices[i][0] is the row number inside X_train / train_df
    idx_teman_rekomendasi = indices[i][0]
    teman = train_df.iloc[idx_teman_rekomendasi]
    
    # Calculate Matches (Logic remains the same)
    is_goal_same = (user['Goal_x'] == teman['Goal_x'])
    is_level_same = (user['level_x'] == teman['level_x'])
    is_freq_same = (user['Workout_Frequency_x'] == teman['Workout_Frequency_x'])
    is_gender_same = (user['Gender_x'] == teman['Gender_x'])
    
    selisih_umur = abs(user['Age_x'] - teman['Age_x'])
    selisih_berat = abs(user['Initial_Weight_kg_x'] - teman['Initial_Weight_kg_x'])
    
    eval_data.append({
        'User_ID': user['User_ID'],
        'User_Goal': user['Goal_x'],
        'Teman_Goal': teman['Goal_x'],
        'Sama_Goal': is_goal_same,
        'Sama_Level': is_level_same,
        'Sama_Freq': is_freq_same,
        'Sama_Gender': is_gender_same,
        'Selisih_Umur': selisih_umur,
        'Selisih_Berat': selisih_berat
    })

df_eval = pd.DataFrame(eval_data)

print("\n" + "="*40)
print("     RECOMMENDATION MODEL REPORT CARD")
print("="*40)

# Key Metrics (Hit Rate)
score_goal = df_eval['Sama_Goal'].mean() * 100
score_level = df_eval['Sama_Level'].mean() * 100
score_freq = df_eval['Sama_Freq'].mean() * 100
score_gender = df_eval['Sama_Gender'].mean() * 100

print(f"\n[A] CATEGORY ACCURACY (Target: >80%)")
print(f"1. Same Goal       : {score_goal:.5f}%")
print(f"2. Same Level      : {score_level:.5f}%")
print(f"3. Same Frequency  : {score_freq:.5f}%")
print(f"4. Same Gender     : {score_gender:.5f}%")

# Numeric Metrics (MAE)
print(f"\n[B] AVERAGE DEVIATION")
print(f"1. Age Diff        : Avg {df_eval['Selisih_Umur'].mean():.5f} years")
print(f"2. Weight Diff     : Avg {df_eval['Selisih_Berat'].mean():.5f} kg")

# Bias Check per Segment
print(f"\n[C] PERFORMANCE BY GOAL TYPE")
segment_group = df_eval.groupby('User_Goal')['Sama_Goal'].agg(['count', 'mean'])
segment_group['Accuracy (%)'] = (segment_group['mean'] * 100).round(2)
print(segment_group[['count', 'Accuracy (%)']])

# Worst Case Analysis
print(f"\n[D] WORST 3 RECOMMENDATIONS (By Weight Diff)")
worst_cases = df_eval.sort_values('Selisih_Berat', ascending=False).head(3)
print(worst_cases[['User_Goal', 'Teman_Goal', 'Selisih_Berat', 'Sama_Level']].to_string(index=False))

print("\n" + "="*40)

Calculating Metric Evaluation...

     RECOMMENDATION MODEL REPORT CARD

[A] CATEGORY ACCURACY (Target: >80%)
1. Same Goal       : 100.00000%
2. Same Level      : 100.00000%
3. Same Frequency  : 73.33333%
4. Same Gender     : 100.00000%

[B] AVERAGE DEVIATION
1. Age Diff        : Avg 4.86667 years
2. Weight Diff     : Avg 9.30000 kg

[C] PERFORMANCE BY GOAL TYPE
             count  Accuracy (%)
User_Goal                       
Muscle Gain     13         100.0
Weight Loss     17         100.0

[D] WORST 3 RECOMMENDATIONS (By Weight Diff)
  User_Goal  Teman_Goal  Selisih_Berat  Sama_Level
Weight Loss Weight Loss             32        True
Weight Loss Weight Loss             22        True
Weight Loss Weight Loss             22        True



### Model Dump

In [279]:
print("Training Final Model on 100% Data...")

# Scale 100% of the User Data
# We create a NEW scaler and fit it on the entire 'df_profiles'.
# This ensures the scaler knows the min/max of ALL your users.
scaler_final = MinMaxScaler()
features_final_scaled = scaler_final.fit_transform(df_profiles[features_knn])

# Train the Matchmaker on 100% Data
# Now we train a NEW KNN model on that complete dataset.
knn_final = NearestNeighbors(n_neighbors=5, metric='euclidean')
knn_final.fit(features_final_scaled)

# Package Everything
# We bundle all the tools your App needs into one dictionary.
data_workout_model = {
    'knn_model': knn_final,         # The Brain (Trained on 100% data)
    'scaler': scaler_final,         # The Tool to scale new user inputs
    'profiles_db': df_profiles,     # The Phonebook (To look up User IDs)
    'schedule_db': df_final,        # The Workout Logs (To see what they did)
    'encoders': {                   # The Translators (Text -> Numbers)
        'gender': le_gender,
        'goal': le_goal,
        'level': le_level
    },
    'features': features_knn        # The Map (Order of inputs)
}

# Save to File
save_path = '../../models/model_workout.pickle'

with open(save_path, 'wb') as f:
    pickle.dump(data_workout_model, f)

print(f"\nSUCCESS: 'model_workout.pickle' saved to {save_path}")

Training Final Model on 100% Data...

SUCCESS: 'model_workout.pickle' saved to ../../models/model_workout.pickle
